# Step 04 — which modules track clinical traits

One correlation per module per trait, then **Benjamini–Hochberg across the whole table**. With 25
modules and 15 traits that is 375 tests; uncorrected, roughly 19 would clear p < 0.05 by chance
alone.

In [ ]:
suppressMessages(library(WGCNA))
w <- readRDS("artifacts/wgcna_A.rds"); e <- readRDS("artifacts/eigengenes_A.rds")
X <- w$X; m <- w$meta; mods <- w$mods; ME <- e$ME

num <- c("SLEDAI_2K","C3_level","C4_level","Duration_years","Lymphocyte_count",
         "uPCR","Creatinine")
ab  <- c("Sm_status","dsDNA_status","La_status","RNP_68_status","RNP_A_status",
         "Ro_52_status","Ro_60_status")
tr <- data.frame(lapply(m[, num], function(v) as.numeric(as.character(v))), row.names = rownames(m))
for (v in ab) tr[[v]] <- as.numeric(m[[v]] == "Positive")

# Age enters as the INDEX of its five-year band, not as an invented exact age.
# The bands are equally spaced, so this gives exactly the same correlation a midpoint
# would -- it just does not claim to know anyone's age.
lvl <- unique(m$Age_group[!is.na(m$Age_group) & m$Age_group != ""])
lvl <- lvl[order(as.numeric(sub("-.*", "", lvl)))]
tr$Age_band <- as.integer(factor(m$Age_group, levels = lvl, ordered = TRUE))

r <- cor(ME, tr, use = "pairwise.complete.obs")
p <- corPvalueStudent(r, nrow(X))
q <- matrix(p.adjust(p, "BH"), nrow = nrow(p), dimnames = dimnames(p))
sprintf("%d tests, %d at FDR 5%%", length(q), sum(q < 0.05))

In [ ]:
h <- which(q < 0.05, arr.ind = TRUE)
res <- data.frame(module = sub("^ME", "", rownames(r)[h[, 1]]),
                  trait = colnames(r)[h[, 2]],
                  r = round(r[h], 2), q = signif(q[h], 2),
                  n_proteins = sapply(sub("^ME", "", rownames(r)[h[, 1]]),
                                      function(k) sum(mods == k)))
head(res[order(-abs(res$r)), ], 15)

## Reading the table honestly

**The interferon module is the most clinically connected thing in the panel**, and it reproduces the
published result on this cohort. Fourteen proteins, nine associations surviving BH correction across
780 tests:

| trait | r | q |
|---|---|---|
| **anti-Sm** | +0.53 | 7.5e-05 |
| C3 | −0.45 | 1.5e-03 |
| age band | −0.45 | 1.6e-03 |
| **anti-Ro60** | +0.40 | 8.7e-03 |
| disease duration | −0.39 | 8.8e-03 |
| anti-dsDNA | +0.39 | 1.2e-02 |
| **anti-RNP-A** | +0.37 | 1.6e-02 |
| **anti-RNP68** | +0.36 | 2.6e-02 |
| lymphocyte count | −0.35 | 3.4e-02 |

The source paper reports that positivity for antibodies against **RNA-binding proteins — anti-Sm,
anti-Ro60, anti-RNP68, anti-RNP-A** — went with increased interferon-stimulated proteins. **All four
are here, and all four survive correction.** Anti-Sm is the single strongest association in the
entire table.

Two things make this more than a coincidence of multiple testing. **Anti-La is flat** — r = +0.16,
q = 0.65 — and it is not on the paper's list, so the one antibody that should *not* track is the one
that does not. And **SLEDAI-2K just misses** at q = 0.082, which is the expected behaviour of a
module tracking an immunological axis rather than a clinical severity score.

**Discount the large modules.** `blue` (1,213 proteins) correlating with creatinine and `black`
(150) with creatinine are eigenproteins over a large fraction of the panel — close to leading
principal components of the whole matrix, where correlating with something is nearly guaranteed. The
interferon module carries its associations on **14 proteins**, which is what makes it interpretable.

**Still open.** This is one fit on one cohort. The module's boundary has moved across earlier draws
of the same patients, so `modulePreservation` across B and C is what decides whether this table
describes biology or this sample.

In [ ]:
sig <- sub("^ME", "", rownames(q)[apply(q, 1, function(z) any(z < 0.05))])
saveRDS(sig, "artifacts/sig_modules_A.rds")
write.csv(r, "artifacts/module_trait_r_A.csv"); write.csv(q, "artifacts/module_trait_q_A.csv")
sprintf("%d of %d modules carry at least one association", length(sig), nrow(q))